In [2]:
import pandas as pd
import numpy as np
import requests as rq
import re

In [243]:
## Overall Objective Identifying broken amazon affilate product link 
## api used : youtube api , amazon : api
## youtube video description of channel id 
## Extracting amazon product link , check the product available of it 


In [244]:
## Using youtube Api and chneel id to extrcct descripition

In [ ]:
API_Key = 'USE_YOUTUBE_API'
channel_id = 'UCXjgbqSZcL2CpTyp-p_7TGQ'

In [245]:
## Step 1 : playlist of channel by using channel id

In [4]:

def get_playlist(channelid):
    url = "https://www.googleapis.com/youtube/v3/channels"
    response = rq.get(url = url , timeout= 10, params={
        "part": "contentDetails",
        "id": channelid,
        "key": API_Key
    } )
    if response.status_code == 200:
        data = response.json()
        playlist = data["items"][0]['contentDetails']['relatedPlaylists']['uploads']
        return playlist
    else: 
        return None

In [5]:
get_playlist(channel_id)

'UUXjgbqSZcL2CpTyp-p_7TGQ'

In [ ]:
## Extraung all channel video using playlist id

In [6]:
def get_videoID(playlistId):
    VideoId = []
    page_token = None
    for page in range(100):
        response = rq.get(
            "https://www.googleapis.com/youtube/v3/playlistItems",
            params={
                "part": "contentDetails",
                "playlistId": playlistId,
                "maxResults": 50,
                "key": API_Key,
                "pageToken": page_token
            }
        )
        if response.status_code != 200:
            return None
        data = response.json()
        for item in data["items"]:
            VideoId.append(item["contentDetails"]["videoId"])
        if "nextPageToken" in data:
            page_token = data["nextPageToken"]
        else:
            break
    return VideoId

In [7]:
Videoid = get_videoID(get_playlist(channel_id))

In [246]:
## Extracting video details using video id

In [8]:
def get_videodata(Videoid):
    video_data = []
    for i in range(0, len(Videoid), 50):

        batch = Videoid[i:i+50]

        response = rq.get(
            "https://www.googleapis.com/youtube/v3/videos",
            params={
                "part": "snippet",
                "id": ",".join(batch),
                "key": API_Key
            }
        )

        if response.status_code == 200:

            data = response.json()

            for video in data["items"]:

                video_data.append({
                    "video_id": video["id"],
                    "published_at": video["snippet"]["publishedAt"],
                    "channel_id": video["snippet"]["channelId"],
                    "title": video["snippet"]["title"],
                    "description": video["snippet"]["description"],
                    "channel_title": video["snippet"]["channelTitle"],
                    "category_id": video["snippet"]["categoryId"]
                })

        else:
            return None
    return video_data

In [9]:
channel_data = get_videodata(get_videoID(get_playlist(channel_id)))

In [247]:
## Extracting all video details of the channel

In [248]:
## Now analyis the chaneel description to identofy amazon links -> product id (asin) -> availability status

In [ ]:
## Analysing the channel data

df = pd.DataFrame(channel_data)

In [ ]:
## Extraing the amazon URL from the data
def findamazonlink(description):
    amazon_links = re.findall(
        r'https?://(?:www\.)?amazon\.[^\s]+',
        description
    )
    return amazon_links

In [190]:
df['amazon_links'] = df.description.apply(findamazonlink)

In [191]:
df["link_count"] = df.amazon_links.apply(len)

In [250]:
df[df["link_count"]>0].shape

(95, 12)

In [251]:
df.shape

(1905, 12)

In [252]:
## Observation 5% video has amazon product link , Total videos = 1905 , amazon product link = 95

In [ ]:
## Now extrating the ASIN from the url 

def extract_asin(links):
    asins = []
    for link in links:
        match = re.search(
            r'(?:/dp/|/gp/product/)([A-Z0-9]{10})',
            link,
            re.IGNORECASE
        )
        if match:
            asins.append(match.group(1))
    return asins


In [194]:
df['amazon_asin'] = df.amazon_links.apply(extract_asin)

In [ ]:
## extracting only the asin in a list

In [195]:
asin_list = df.amazon_asin.to_list()

In [196]:
values  = []
for i in asin_list:
    for k in i:
        values.append(k)

In [ ]:
## Using searchapi API to identify the availablty of the product in amazon

In [ ]:
import requests

def find_asin(asin):
    url = "https://www.searchapi.io/api/v1/search"
    params = {
    "engine": "amazon_product",
    "asin": asin,
    "api_key": "Use_searchapi",
    'amazon_domain':'amazon.in'
    }

    response = requests.get(url, params=params, timeout = 30)
    if response.status_code == 200:
            return response.json()
    else:
          return None



In [253]:
## Getting all Possible data from the api 

In [80]:
result1 = []

for i in values:
    status = find_asin(i)
    result1.append({"asin":i , "json":status})

In [ ]:
## Now identify statud and detailed status of the product Asin

In [ ]:
def asin_status(result):
    asin_status = []
    for item in result:
        asin = item.get('asin')
        data = item.get("json")
        if not data or not data.get("product"):
            status = "Product not found"

        else:
            product = data["product"]
            buybox = product.get("buybox")
            if not buybox:
                status = "No buy box"

            else:
                availability = buybox.get("availability", "").lower()

                if availability.find("in stock") != -1:
                    status = "In stock"

                elif availability == "currently unavailable.":
                    status = "Out of stock"
                else:
                    status = "Unknown"
                if status == "In stock":
                    final_status = "available"
                else:
                    final_status = "unavailable"
        asin_status.append({"asin" : asin , "detailed_status" : status, "status" : final_status})
    return asin_status

In [212]:
## checking the overall status of the product

product_status = asin_status(result1)

In [ ]:
## Converting the data into data farme for anlayis

df_status = pd.DataFrame(product_status)

In [259]:
Total_asin = df_status["asin"].shape[0]
unavaible_asin = df_status[df_status["status"]=="unavailable"].shape[0]
percentage = (unavaible_asin/Total_asin)*100
print(f'Total_asin = {Total_asin}  , unavaible_asin = {unavaible_asin} , invaild_percent = {percentage}')

Total_asin = 59  , unavaible_asin = 42 , invaild_percent = 71.1864406779661


In [ ]:
### Now mapping the video data back to vidoes , for each asin and there status

In [ ]:
## Identify result of asin
def asin_result(asin):
    status = df_status[df_status["asin"]==asin].status.iloc[0]
    return status

In [263]:
asin_result("B01L5NTYSO")

'unavailable'

In [229]:
## Now link the asin staus back to asin
def asin_list(asin_list):
    result = []
    if len(asin_list) > 0:
        for i in asin_list:
            result.append(asin_result(i))
    return result

In [230]:
df['asin_status'] = df["amazon_asin"].apply(asin_list)

In [232]:
def broken_video_link(asin_status):
    if 'unavailable' in asin_status:
        return "broken_afffilate_link"
    else:
        return None

In [ ]:
## identify broken affilate liks in videos
df["affilate_status"] = df.asin_status.apply(broken_video_link)

In [271]:
## Observation
Total_video = df.video_id.nunique()
amazon_affilate_video = df[df["link_count"]>0].video_id.nunique()
broken_affilate_Video = df[df["affilate_status"]=="broken_afffilate_link"].video_id.nunique()
broken_affilate_per = (broken_affilate_Video/amazon_affilate_video ) *100

In [272]:
print(f'Total_video = {Total_video}  , amazon_affilate_video = {amazon_affilate_video} , broken_affilate_Video = {broken_affilate_Video} , broken_affilate_per = {broken_affilate_per} ')

Total_video = 1905  , amazon_affilate_video = 95 , broken_affilate_Video = 8 , broken_affilate_per = 8.421052631578947 
